# Nedbank Colab Search

Objective:
- Reproduce the strongest rolling-panel public-feedback branch on stronger hardware.
- Re-run the public calibration branch with more compute headroom.
- Export the next upload-ready CSV pack from the same deterministic codebase.


## Notes

- Use a CPU runtime with High-RAM if available. These scripts are CPU-bound; GPU does not materially help.
- Keep the repo and competition data together in one Google Drive folder.
- This notebook assumes the project folder already contains the repo files and the extracted competition data.


In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

os.environ['LOKY_MAX_CPU_COUNT'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

PROJECT_DIR = Path('/content/drive/MyDrive/Nedbank Transaction Volume Forecasting Challenge')
assert PROJECT_DIR.exists(), f'Update PROJECT_DIR first: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print(PROJECT_DIR)
print(sorted(p.name for p in PROJECT_DIR.iterdir())[:20])

In [ ]:
required_paths = [
    Path('Train.csv'),
    Path('Test.csv'),
    Path('SampleSubmission.csv'),
    Path('transactions_features/transactions_features.parquet'),
    Path('financials_features/financials_features.parquet'),
    Path('demographics_clean/demographics_clean.parquet'),
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f'Missing required files: {missing}'
missing

## Rebuild The Rolling-Panel Branch

This regenerates the anchored rolling-panel XGBoost branch that produced the current best public family.

In [ ]:
rolling_cmd = [
    'python',
    'rolling_panel_supervision_lab.py',
    '--data-dir', '.',
    '--run-name', 'colab_rolling_panel_xgb_strict',
    '--include-models', 'xgb_conservative_v3',
    '--n-splits', '5',
    '--n-repeats', '1',
    '--pseudo-cutoff-start', '2014-08',
    '--pseudo-cutoff-end', '2015-07',
]
subprocess.run(rolling_cmd, check=True)

## Run The Stability Selector

This is the low-noise public-proxy search that rebuilds the stability branch.

In [ ]:
stability_cmd = [
    'python',
    'rolling_panel_public_stability_lab.py',
    '--data-dir', '.',
    '--run-name', 'colab_public_stability_strict',
    '--pseudo-public-splits', '240',
]
subprocess.run(stability_cmd, check=True)

## Run The Calibration Selector

This is the current highest-priority late-stage branch. It applies smooth isotonic and log-space recalibration around the last confirmed public-best file.

In [ ]:
calibration_cmd = [
    'python',
    'rolling_panel_public_calibration_lab.py',
    '--data-dir', '.',
    '--run-name', 'colab_public_calibration',
]
subprocess.run(calibration_cmd, check=True)

In [ ]:
from pathlib import Path

calibration_root = Path('outputs/rolling_panel_public_calibration_lab')
latest_run = max(calibration_root.iterdir(), key=lambda path: path.stat().st_mtime)
print('Latest run:', latest_run)
print((latest_run / 'summary.md').read_text(encoding='utf-8'))

In [ ]:
recommended_dir = latest_run / 'recommended_selection'
print('Recommended files:')
for path in sorted(recommended_dir.glob('*.csv')):
    print(path)
print('\nREADME:\n')
print((recommended_dir / 'README.md').read_text(encoding='utf-8'))

## Next Step

Upload the first CSV from the latest `recommended_selection` folder. If it improves, try the second. If it does not improve, prefer the lower-drift hedge before spending another submission.